# Trade EDA Solution

This notebook is the **solution version** for the trade EDA and visualization exercises.


In [ ]:
import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
pd.set_option("display.max_columns", 60)
plt.style.use("default")


In [ ]:
PROJECT_ROOT = os.getcwd()
if not os.path.isdir(os.path.join(PROJECT_ROOT, "data", "trade")):
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

trade_path  = Path('../../data/0_raw/malawi/trade data/Trade Data Python Training.xlsx')

df_raw = pd.read_excel(trade_path)
df = df_raw.copy()
df.columns = [c.strip().lower() for c in df.columns]

required_cols = ["year", "period", "flow", "partner", "hscode", "description", "mwkvalue"]
absent_required = [c for c in required_cols if c not in df.columns]
if absent_required:
    raise ValueError(f"Required columns not found: {absent_required}")

df.head()


## 1) First diagnostics + univariate EDA


In [ ]:
month_map = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
    "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12
}

df_work = df.copy()
df_work["flow"] = df_work["flow"].astype(str).str.upper().str.strip()
df_work["year_num"] = pd.to_numeric(df_work["year"], errors="coerce").astype("Int64")

period_num = pd.to_numeric(df_work["period"], errors="coerce")
period_text = df_work["period"].astype(str).str.strip().str[:3].str.lower()
period_num = period_num.fillna(period_text.map(month_map))
df_work["period_num"] = period_num.astype("Int64")

df_work["mwkvalue_num"] = pd.to_numeric(df_work["mwkvalue"], errors="coerce")

dtypes = df_work.dtypes.rename("dtype").to_frame()
row_key = ["year", "period", "flow", "partner", "hscode", "description", "mwkvalue"]
duplicate_rows = int(df_work.duplicated(subset=row_key).sum())

valid_flows = {"I", "E", "RE", "R"}
invalid_flow_mask = ~df_work["flow"].isin(valid_flows)
invalid_period_mask = ~(df_work["period_num"].between(1, 12)) | (df_work["period_num"].isna())
invalid_mwk_numeric_mask = df_work["mwkvalue_num"].isna()
negative_mwk_mask = df_work["mwkvalue_num"] < 0

print("Duplicate rows on row key:", duplicate_rows)
print("Invalid flow rows:", int(invalid_flow_mask.sum()))
print("Invalid period rows:", int(invalid_period_mask.sum()))
print("Non-numeric MWKValue rows:", int(invalid_mwk_numeric_mask.sum()))
print("Negative MWKValue rows:", int(negative_mwk_mask.sum()))

display(dtypes.head(15))


In [ ]:
mwk_clean = df_work["mwkvalue_num"].dropna()
eda_summary = pd.DataFrame([
    {
        "count": int(mwk_clean.shape[0]),
        "min": float(mwk_clean.min()) if len(mwk_clean) else np.nan,
        "max": float(mwk_clean.max()) if len(mwk_clean) else np.nan,
        "p50": float(mwk_clean.quantile(0.50)) if len(mwk_clean) else np.nan,
        "p95": float(mwk_clean.quantile(0.95)) if len(mwk_clean) else np.nan,
        "p99": float(mwk_clean.quantile(0.99)) if len(mwk_clean) else np.nan,
    }
])

issue_register = pd.DataFrame([
    {"issue": "Invalid flow codes", "count": int(invalid_flow_mask.sum()), "action": "Review or map unexpected flow values"},
    {"issue": "Invalid period values", "count": int(invalid_period_mask.sum()), "action": "Fix month parsing or remove invalid rows"},
    {"issue": "Non-numeric MWKValue", "count": int(invalid_mwk_numeric_mask.sum()), "action": "Coerce/clean value field before analysis"},
    {"issue": "Negative MWKValue", "count": int(negative_mwk_mask.sum()), "action": "Validate sign and coding conventions"},
    {"issue": "Duplicate row key", "count": duplicate_rows, "action": "Deduplicate or verify repeated records"},
])

display(eda_summary)
display(issue_register)


In [ ]:
plot_data = df_work[df_work["mwkvalue_num"].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(plot_data["mwkvalue_num"], bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("MWKValue distribution (overall)")
axes[0].set_xlabel("MWKValue")
axes[0].set_ylabel("Count")

axes[1].boxplot(plot_data["mwkvalue_num"].dropna(), vert=False)
axes[1].set_title("MWKValue boxplot (overall)")
axes[1].set_xlabel("MWKValue")
plt.tight_layout()
plt.show()

flow_order = [f for f in ["I", "E", "RE", "R"] if f in plot_data["flow"].dropna().unique()]
flow_data = [plot_data.loc[plot_data["flow"] == f, "mwkvalue_num"].dropna() for f in flow_order]

plt.figure(figsize=(8, 4))
plt.boxplot(flow_data, labels=flow_order)
plt.title("MWKValue by Flow")
plt.xlabel("Flow")
plt.ylabel("MWKValue")
plt.tight_layout()
plt.show()

top_partners = (
    plot_data.groupby("partner", as_index=False)["mwkvalue_num"]
    .sum()
    .sort_values("mwkvalue_num", ascending=False)
    .head(10)
    .sort_values("mwkvalue_num", ascending=True)
)

plt.figure(figsize=(10, 5))
plt.barh(top_partners["partner"], top_partners["mwkvalue_num"], color="seagreen")
plt.title("Top 10 partners by total MWKValue")
plt.xlabel("Total MWKValue")
plt.ylabel("Partner")
plt.tight_layout()
plt.show()

top_hs = (
    plot_data.groupby("hscode", as_index=False)["mwkvalue_num"]
    .sum()
    .sort_values("mwkvalue_num", ascending=False)
    .head(10)
    .sort_values("mwkvalue_num", ascending=True)
)

plt.figure(figsize=(10, 5))
plt.barh(top_hs["hscode"].astype(str), top_hs["mwkvalue_num"], color="royalblue")
plt.title("Top 10 HS codes by total MWKValue")
plt.xlabel("Total MWKValue")
plt.ylabel("HScode")
plt.tight_layout()
plt.show()


In [ ]:
df_work[df_work['mwkvalue'] > (df_work['mwkvalue'].max() - 10**5) ]

## 2) Aggregation + pivot tables


In [ ]:
valid_rows = (
    df_work["year_num"].notna()
    & df_work["period_num"].between(1, 12)
    & df_work["mwkvalue_num"].notna()
)
trade = df_work.loc[valid_rows].copy()

trade["date"] = pd.to_datetime(
    trade["year_num"].astype(str) + "-" + trade["period_num"].astype(str).str.zfill(2) + "-01",
    errors="coerce"
)

flow_sign = {"I": -1, "E": 1, "RE": 1, "R": -1}
trade["flow_sign"] = trade["flow"].map(flow_sign)
trade["signed_mwk"] = trade["mwkvalue_num"] * trade["flow_sign"]

imports_exports = (
    trade[trade["flow"].isin(["I", "E"])]
    .groupby(["date", "flow"], as_index=False)["mwkvalue_num"]
    .sum()
    .pivot(index="date", columns="flow", values="mwkvalue_num")
    .fillna(0)
    .sort_index()
)

if "I" not in imports_exports.columns:
    imports_exports["I"] = 0.0
if "E" not in imports_exports.columns:
    imports_exports["E"] = 0.0

imports_exports["trade_balance_E_minus_I"] = imports_exports["E"] - imports_exports["I"]
imports_exports = imports_exports.reset_index()

display(imports_exports.head())

partner_flow = pd.pivot_table(
    trade,
    index="partner",
    columns="flow",
    values="mwkvalue_num",
    aggfunc="sum",
    fill_value=0,
)
partner_flow["total"] = partner_flow.sum(axis=1)
partner_flow = partner_flow.sort_values("total", ascending=False)

hscode_flow = pd.pivot_table(
    trade,
    index="hscode",
    columns="flow",
    values="mwkvalue_num",
    aggfunc="sum",
    fill_value=0,
)
hscode_flow["total"] = hscode_flow.sum(axis=1)
hscode_flow = hscode_flow.sort_values("total", ascending=False).head(10)

display(partner_flow.head(10))
display(hscode_flow)

output_dir = os.path.join(PROJECT_ROOT, "data", "trade", "1_intermediate")
os.makedirs(output_dir, exist_ok=True)
partner_flow.to_csv(os.path.join(output_dir, "trade_by_partner_flow.csv"))
hscode_flow.to_csv(os.path.join(output_dir, "trade_by_hscode_flow.csv"))

print("Saved:")
print("-", os.path.join(output_dir, "trade_by_partner_flow.csv"))
print("-", os.path.join(output_dir, "trade_by_hscode_flow.csv"))

top10_partners_by_year = (
    trade.groupby(["year_num", "partner"], as_index=False)["mwkvalue_num"]
    .sum()
    .sort_values(["year_num", "mwkvalue_num"], ascending=[True, False])
    .groupby("year_num")
    .head(10)
)
display(top10_partners_by_year.head(20))


## 3) Bivariate EDA: Seasonality by Flow + Partner Concentration


In [ ]:
seasonality = (
    trade.groupby(["flow", "period_num"], as_index=False)["mwkvalue_num"]
    .sum()
    .sort_values(["flow", "period_num"])
)

plt.figure(figsize=(10, 5))
for flow_code in ["I", "E", "RE", "R"]:
    tmp = seasonality[seasonality["flow"] == flow_code]
    if len(tmp) > 0:
        plt.plot(tmp["period_num"], tmp["mwkvalue_num"], marker="o", label=flow_code)

plt.title("Seasonality by Flow (Monthly Totals, Aggregated Across Years)")
plt.xlabel("Month")
plt.ylabel("Total MWKValue")
plt.xticks(range(1, 13))
plt.legend(title="Flow")
plt.tight_layout()
plt.show()

seasonality_year = (
    trade.groupby(["flow", "year_num", "period_num"], as_index=False)["mwkvalue_num"]
    .sum()
)

flows = [f for f in ["I", "E", "RE", "R"] if f in seasonality_year["flow"].unique()]
fig, axes = plt.subplots(len(flows), 1, figsize=(11, 4 * max(1, len(flows))), sharex=True)
if len(flows) == 1:
    axes = [axes]

for ax, flow_code in zip(axes, flows):
    tmp = seasonality_year[seasonality_year["flow"] == flow_code]
    years = sorted([y for y in tmp["year_num"].dropna().unique()])
    for yr in years:
        yr_tmp = tmp[tmp["year_num"] == yr]
        ax.plot(yr_tmp["period_num"], yr_tmp["mwkvalue_num"], marker="o", label=str(int(yr)))
    ax.set_title(f"Seasonality by Flow and Year - {flow_code}")
    ax.set_xlabel("Month")
    ax.set_ylabel("Total MWKValue")
    ax.set_xticks(range(1, 13))
    ax.legend(title="Year")

plt.tight_layout()
plt.show()


def partner_concentration_table(data, flow_code):
    tmp = data[data["flow"] == flow_code].copy()
    grouped = (
        tmp.groupby(["year_num", "partner"], as_index=False)["mwkvalue_num"]
        .sum()
        .sort_values(["year_num", "mwkvalue_num"], ascending=[True, False])
    )

    records = []
    for year, g in grouped.groupby("year_num"):
        total = g["mwkvalue_num"].sum()
        if total <= 0:
            records.append(
                {
                    "year": int(year),
                    "top1_share": np.nan,
                    "top3_share": np.nan,
                    "top5_share": np.nan,
                    "hhi": np.nan,
                }
            )
            continue

        shares = g["mwkvalue_num"] / total
        records.append(
            {
                "year": int(year),
                "top1_share": float(shares.head(1).sum()),
                "top3_share": float(shares.head(3).sum()),
                "top5_share": float(shares.head(5).sum()),
                "hhi": float((shares ** 2).sum()),
            }
        )

    return pd.DataFrame(records).sort_values("year")


concentration_imports = partner_concentration_table(trade, "I")
concentration_exports = partner_concentration_table(trade, "E")

print("Partner concentration - Imports")
display(concentration_imports)
print("Partner concentration - Exports")
display(concentration_exports)


## 4) Visualization clinic (Before/After)


In [ ]:
exports_time = (
    trade[trade["flow"] == "E"]
    .groupby("date", as_index=False)["mwkvalue_num"]
    .sum()
    .sort_values("date")
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(exports_time["date"], exports_time["mwkvalue_num"], color="tomato")
ymin = exports_time["mwkvalue_num"].min() * 0.95 if len(exports_time) else 0
axes[0].set_ylim(ymin, exports_time["mwkvalue_num"].max() * 1.02 if len(exports_time) else 1)
axes[0].set_title("Before: Truncated-axis bar chart")
axes[0].tick_params(axis="x", rotation=45)

axes[1].plot(exports_time["date"], exports_time["mwkvalue_num"], marker="o", color="steelblue")
axes[1].set_ylim(0, exports_time["mwkvalue_num"].max() * 1.05 if len(exports_time) else 1)
axes[1].set_title("After: Line chart with proper axis")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
exports_partner = (
    trade[trade["flow"] == "E"]
    .groupby("partner", as_index=False)["mwkvalue_num"]
    .sum()
    .sort_values("mwkvalue_num", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

pie_data = exports_partner.head(20)
axes[0].pie(pie_data["mwkvalue_num"], labels=pie_data["partner"], startangle=90)
axes[0].set_title("Before: Pie chart with many categories")

top10 = exports_partner.head(10).copy()
other_value = exports_partner.iloc[10:]["mwkvalue_num"].sum()
if other_value > 0:
    top10 = pd.concat(
        [top10, pd.DataFrame([{"partner": "Other", "mwkvalue_num": other_value}])],
        ignore_index=True,
    )

top10 = top10.sort_values("mwkvalue_num", ascending=True)
axes[1].barh(top10["partner"], top10["mwkvalue_num"], color="seagreen")
axes[1].set_title("After: Top 10 partners + Other")
axes[1].set_xlabel("Total export MWKValue")

plt.tight_layout()
plt.show()


In [ ]:
imports_hs = (
    trade[trade["flow"] == "I"]
    .groupby(["hscode", "description"], as_index=False)["mwkvalue_num"]
    .sum()
    .sort_values("mwkvalue_num", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bad = imports_hs.head(10).sample(frac=1, random_state=42)
axes[0].bar(bad["description"], bad["mwkvalue_num"], color="gray")
axes[0].set_title("Before: Unsorted vertical bars")
axes[0].tick_params(axis="x", rotation=75)

best = imports_hs.head(10).copy()
best["label"] = best.apply(
    lambda r: f"{r['hscode']} - " + "\n".join(textwrap.wrap(str(r["description"]), width=26)[:2]),
    axis=1,
)
best = best.sort_values("mwkvalue_num", ascending=True)
axes[1].barh(best["label"], best["mwkvalue_num"], color="royalblue")
axes[1].set_title("After: Sorted horizontal bars with readable labels")
axes[1].set_xlabel("Total import MWKValue")

plt.tight_layout()
plt.show()


## 5) Sankey diagram for trade flows


In [ ]:
import plotly.graph_objects as go

latest_year = int(trade["year_num"].dropna().max())
sankey_base = trade[trade["year_num"] == latest_year].copy()

top_n = 5
top_partners = (
    sankey_base.groupby("partner", as_index=False)["mwkvalue_num"]
    .sum()
    .sort_values("mwkvalue_num", ascending=False)
    .head(top_n)["partner"]
    .tolist()
)

sankey_base["partner_group"] = np.where(sankey_base["partner"].isin(top_partners), sankey_base["partner"], "Other")

links_df = (
    sankey_base.groupby(["partner_group", "flow"], as_index=False)["mwkvalue_num"]
    .sum()
)

source_nodes = links_df["partner_group"].drop_duplicates().tolist()
target_nodes = links_df["flow"].drop_duplicates().tolist()
all_nodes = source_nodes + target_nodes

node_index = {name: i for i, name in enumerate(all_nodes)}

source = links_df["partner_group"].map(node_index).tolist()
target = links_df["flow"].map(node_index).tolist()
value = links_df["mwkvalue_num"].tolist()

fig = go.Figure(
    data=[
        go.Sankey(
            node=dict(label=all_nodes, pad=20, thickness=18),
            link=dict(source=source, target=target, value=value),
        )
    ]
)
fig.update_layout(title_text=f"Trade Sankey (Year {latest_year})", font_size=11)
fig.show()


## 6) Mini-capstone: 1-page briefing outputs


In [ ]:
annual_flow = (
    trade[trade["flow"].isin(["I", "E"])]
    .groupby(["year_num", "flow"], as_index=False)["mwkvalue_num"]
    .sum()
    .pivot(index="year_num", columns="flow", values="mwkvalue_num")
    .fillna(0)
)

if "I" not in annual_flow.columns:
    annual_flow["I"] = 0.0
if "E" not in annual_flow.columns:
    annual_flow["E"] = 0.0

annual_flow["trade_balance_E_minus_I"] = annual_flow["E"] - annual_flow["I"]
annual_flow = annual_flow.reset_index().rename(columns={"year_num": "year"})

exports_partner_year = (
    trade[trade["flow"] == "E"]
    .groupby(["year_num", "partner"], as_index=False)["mwkvalue_num"]
    .sum()
    .sort_values(["year_num", "mwkvalue_num"], ascending=[True, False])
)

top5_export_partners = exports_partner_year.groupby("year_num").head(5).copy()
combined_share_records = []
for year, grp in exports_partner_year.groupby("year_num"):
    total = grp["mwkvalue_num"].sum()
    top5_total = grp.head(5)["mwkvalue_num"].sum()
    combined_share_records.append(
        {
            "year": int(year),
            "top5_export_share": (top5_total / total) if total > 0 else np.nan,
        }
    )
combined_top5_share = pd.DataFrame(combined_share_records).sort_values("year")

imports_hs_year = (
    trade[trade["flow"] == "I"]
    .groupby(["year_num", "hscode", "description"], as_index=False)["mwkvalue_num"]
    .sum()
    .sort_values(["year_num", "mwkvalue_num"], ascending=[True, False])
)
top5_import_hs = imports_hs_year.groupby("year_num").head(5)

print("Headline indicators:")
display(annual_flow)
print("Top 5 export partner combined share:")
display(combined_top5_share)
print("Top 5 export partners (detail):")
display(top5_export_partners.head(20))
print("Top 5 import HS codes:")
display(top5_import_hs.head(20))


In [ ]:
plt.figure(figsize=(10, 5))
annual_plot = annual_flow.sort_values("year")
plt.plot(annual_plot["year"], annual_plot["I"], marker="o", label="Imports")
plt.plot(annual_plot["year"], annual_plot["E"], marker="o", label="Exports")
plt.plot(annual_plot["year"], annual_plot["trade_balance_E_minus_I"], marker="o", label="Balance (E-I)")
plt.title("Trade headline indicators by year")
plt.xlabel("Year")
plt.ylabel("MWKValue")
plt.legend()
plt.tight_layout()
plt.show()

latest_year = int(trade["year_num"].dropna().max())
latest_top_export = (
    trade[(trade["year_num"] == latest_year) & (trade["flow"] == "E")]
    .groupby("partner", as_index=False)["mwkvalue_num"]
    .sum()
    .sort_values("mwkvalue_num", ascending=False)
    .head(10)
    .sort_values("mwkvalue_num", ascending=True)
)

plt.figure(figsize=(10, 5))
plt.barh(latest_top_export["partner"], latest_top_export["mwkvalue_num"], color="darkcyan")
plt.title(f"Top export partners - {latest_year}")
plt.xlabel("MWKValue")
plt.ylabel("Partner")
plt.tight_layout()
plt.show()

summary_note = (
"""
### EDA summary note
- Data was profiled for types, duplicate keys, and domain validation checks.
- Core indicators were built for imports, exports, and trade balance by year.
- Partner concentration and seasonal patterns were reviewed to identify plausibility and risk areas.
- Latest-year export structure appears concentrated in top partners; details are in the partner tables above.
"""
)
print(summary_note)
